# Capítulo 5 — Pré-recuperação (Pre-Retrieval Processing)

**Livro:** começa na página **250** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Text-to-SQL: Transforming natural language into SQL · PDF p. 253

#### Exercício: Pergunta em português vira SQL

Crie no `sqlite3` a tabela `emprestimos(leitor, livro, data_saida, devolvido)` com algumas linhas. Peça ao modelo o SQL para "quantos livros ainda não foram devolvidos?" (passe o esquema no *prompt* e peça só o SQL), execute e mostre.

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE emprestimos (leitor TEXT, livro TEXT, data_saida TEXT, devolvido INTEGER)")
con.executemany("INSERT INTO emprestimos VALUES (?,?,?,?)", [("ana", "Dom Casmurro", "2025-05-01", 1), ("bia", "Vidas Secas", "2025-05-03", 0),
    ("caio", "O Cortiço", "2025-05-04", 0), ("ana", "Iracema", "2025-05-10", 0)])
esquema = "emprestimos(leitor TEXT, livro TEXT, data_saida TEXT, devolvido INTEGER 0/1)"
sql = perguntar(f"Esquema SQLite: {esquema}\nEscreva só o SQL, sem explicação e sem ```, para: quantos livros ainda não foram devolvidos?")
sql = sql.strip().strip("`").removeprefix("sql").strip()
print(sql); print(con.execute(sql).fetchall())

## Self-query retriever automatically generating metadata filter conditions from queries · PDF p. 276

#### Exercício: O modelo cria o filtro de metadados

Peça ao modelo para transformar "regras de multa atualizadas em 2025" num JSON `{"consulta": ..., "data_minima": ...}`. Aplique o filtro de data nos `DOCS` e faça a busca por palavras só nos que passaram.

In [ ]:
pedido = "regras de multa atualizadas em 2025"
resp = perguntar(f'Converta em JSON com as chaves "consulta" (palavras-chave) e "data_minima" (AAAA-MM-DD ou null). Só o JSON.\nPedido: {pedido}',
                 response_format={"type": "json_object"})
f = json.loads(resp); print(f)
cands = [d for d in DOCS if not f.get("data_minima") or d["data"] >= f["data_minima"]]
qs = set(re.findall(r"\w+", f["consulta"].lower()))
print(max(cands, key=lambda d: len(qs & set(re.findall(r"\w+", d["texto"].lower()))))["titulo"])

## Query translation: Better explaining user questions · PDF p. 284

#### Exercício: Reescrita de consulta e multi-query

Para a pergunta vaga "e se eu esquecer de devolver?", peça ao modelo 3 reformulações diferentes. Busque com cada uma (BM25) e junte os resultados sem repetição.

In [ ]:
from rank_bm25 import BM25Okapi
bm = BM25Okapi([re.findall(r"\w+", d["texto"].lower()) for d in DOCS])
vars_ = perguntar("Reescreva de 3 formas diferentes, uma por linha, sem numerar: e se eu esquecer de devolver o livro da biblioteca?").splitlines()
achados = []
for v in [x for x in vars_ if x.strip()]:
    i = int(np.argmax(bm.get_scores(re.findall(r"\w+", v.lower()))))
    if DOCS[i]["titulo"] not in achados: achados.append(DOCS[i]["titulo"])
print(vars_); print(achados)

## Query routing finding the right data source · PDF p. 300

#### Exercício: Roteador de consultas

Peça ao modelo para classificar cada pergunta em `"sql"` (dados de empréstimos), `"documentos"` (regras da biblioteca) ou `"conversa"` (cumprimentos), respondendo só com a palavra. Teste com 3 perguntas.

In [ ]:
def rotear(q):
    r = perguntar(f'Classifique em sql, documentos ou conversa. Responda só a palavra.\n- sql: números sobre empréstimos\n- documentos: regras e serviços da biblioteca\n- conversa: cumprimentos\nPergunta: {q}', temperature=0)
    return r.strip().lower().strip(".")
for q in ["quantos empréstimos a Ana fez?", "a sala de estudos tem tomada?", "oi, tudo bem?"]:
    print(q, "->", rotear(q))